# Trening YOLOv8n - SoccerNet (ball, player, referee)

Trening modelu YOLOv8n do detekcji obiektow na obrazach z meczu pilki noznej.
Srodowisko docelowe: Google Colab z GPU (L4 zalecane, High-RAM nie wymagany).
Dane wejsciowe: archiwum w formacie YOLO przechowywane na Google Drive.


## 1. Konfiguracja srodowiska


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

import os, shutil, glob, torch
from ultralytics import YOLO

# Weryfikacja dostepnego akceleratora
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

assert gpu_mem > 15, "Niewystarczajaca pamiec GPU - wymagane co najmniej 16 GB VRAM"


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: NVIDIA L4 (23.7 GB)
PyTorch: 2.10.0+cu128, CUDA: 12.8


## 2. Sciezki i stale projektu

Centralne definicje sciezek i nazw - zmiana w jednym miejscu propaguje sie przez caly notebook.


In [ ]:
ZIP_PATH       = "/content/drive/MyDrive/CvFootballTracker_Data/Detection/yoloformat.zip"
LOCAL_DATA_DIR = "/content/yoloformat"
PROJECT_DIR    = "/content/drive/MyDrive/CvFootballTracker_Data/results"
RUN_NAME       = "yolo_soccernet_n_v1"


## 3. Ekstrakcja danych z Google Drive

Krok wykonywany jednorazowo - przy ponownym uruchomieniu komorki dane sa pomijane jezeli juz istnieja na dysku lokalnym VM.


In [ ]:
if not os.path.exists(LOCAL_DATA_DIR):
    print("Rozpakowywanie archiwum...")
    shutil.unpack_archive(ZIP_PATH, extract_dir="/content/")
    print("Zakonczono.")
else:
    print("Dane juz dostepne na dysku lokalnym.")

n_train = len(glob.glob(f"{LOCAL_DATA_DIR}/train/*/images/*.jpg"))
n_valid = len(glob.glob(f"{LOCAL_DATA_DIR}/valid/*/images/*.jpg"))
print(f"Liczba obrazow - train: {n_train}, valid: {n_valid}")


Rozpakowywanie archiwum...
Zakonczono.
Liczba obrazow - train: 11500, valid: 8250


## 4. Generacja pliku konfiguracyjnego data.yaml

Sciezki w standardzie POSIX, niezalezne od srodowiska zrodlowego (Windows lokalnie vs Linux na Colab).


In [ ]:
yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: train
val: valid
test: test

names:
  0: ball
  1: player
  2: referee
"""
with open(f"{LOCAL_DATA_DIR}/data.yaml", 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())

print(f"Zapisano: {LOCAL_DATA_DIR}/data.yaml")


Zapisano: /content/yoloformat/data.yaml


## 5. Inicjalizacja modelu z obsluga wznawiania

Jezeli wczesniejszy trening zostal przerwany (np. timeout sesji Colab), automatyczne podjecie z ostatniego checkpointu zapisanego na Drive.


In [ ]:
last_ckpt = f"{PROJECT_DIR}/{RUN_NAME}/weights/last.pt"

if os.path.exists(last_ckpt):
    print(f"Wykryto checkpoint, wznawianie z: {last_ckpt}")
    model = YOLO(last_ckpt)
    resume = True
else:
    print("Inicjalizacja od wag pretrenowanych na zbiorze COCO")
    model = YOLO("yolov8n.pt")
    resume = False


Inicjalizacja od wag pretrenowanych na zbiorze COCO


## 6. Trening

Glowna petla treningowa. Czas wykonania na L4: okolo 6 godzin dla 100 epok. Wyniki zapisywane na biezaco do `PROJECT_DIR/RUN_NAME` na Google Drive.


In [ ]:
results = model.train(
    # Dane wejsciowe
    data=f"{LOCAL_DATA_DIR}/data.yaml",

    # Rozdzielczosc i wielkosc batcha
    imgsz=960,              # wyzsza rozdzielczosc poprawia detekcje malych obiektow
    batch=32,

    # Czas trwania treningu
    epochs=100,
    patience=30,            # early stopping przy braku poprawy mAP50 przez N epok

    # Harmonogram learning rate
    cos_lr=True,            # cosine annealing
    warmup_epochs=3,

    # Wydajnosc
    cache='ram',            # cache obrazow w RAM
    workers=8,
    amp=True,               # automatic mixed precision (FP16)
    device=0,

    # Parametry augmentacji
    mosaic=1.0,
    close_mosaic=15,        # wylaczenie mosaic w koncowych epokach dla stabilizacji
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,            # brak rotacji - stala perspektywa kamery boiska
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,             # bez odbicia pionowego
    fliplr=0.5,             # odbicie poziome - boisko symetryczne

    # Zapis wynikow
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    save_period=-1,         # zapis wylacznie best.pt i last.pt
    resume=resume,

    # Reprodukowalnosc
    seed=17,
    deterministic=True,

    # Logowanie
    plots=True,
    verbose=True,
)


Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yoloformat/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_soccernet_n_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

## 7. Walidacja koncowa i raport metryk per klasa

Ocena najlepszego modelu (best.pt) na zbiorze walidacyjnym. Raport zawiera metryki ogolne oraz rozbicie per klasa.


In [ ]:
print("="*60)
print("Walidacja koncowa (best.pt)")
print("="*60)

best_path = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"
best_model = YOLO(best_path)
metrics = best_model.val(data=f"{LOCAL_DATA_DIR}/data.yaml", split='val', imgsz=960)

print(f"\nmAP50 ogolne:    {metrics.box.map50:.4f}")
print(f"mAP50-95 ogolne: {metrics.box.map:.4f}")
print(f"\nmAP50-95 per klasa:")
for i, name in enumerate(['ball', 'player', 'referee']):
    print(f"  {name:10s} {metrics.box.maps[i]:.4f}")

print(f"\nSciezka do najlepszego modelu: {best_path}")


Walidacja koncowa (best.pt)
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2707.3±616.0 MB/s, size: 250.6 KB)
val: Scanning /content/yoloformat/valid/SNMOT-160/labels... 8250 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8250/8250 1.4Kit/s 5.7s
val: New cache created: /content/yoloformat/valid/SNMOT-160/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 516/516 8.5it/s 1:01
                   all       8250     135267      0.848      0.795      0.792      0.457
                  ball       7833       8241       0.65      0.471       0.44      0.141
                player       8250     114663      0.955      0.965       0.97      0.627
               referee       7685      12363       0.94       0.95      0.967      0.603
Speed: 0.9ms preprocess,

---

## Komorka opcjonalna - sprzatanie

Usuwa folder wynikowy danego treningu z Google Drive. Uzywac swiadomie - operacja nieodwracalna. Przydatne gdy chcesz wystartowac dany RUN_NAME od zera.


In [ ]:
target = f"{PROJECT_DIR}/{RUN_NAME}"
if os.path.exists(target):
    confirm = input(f"Usunac {target}? (tak/nie): ")
    if confirm.lower() == "tak":
        shutil.rmtree(target)
        print("Usunieto.")
    else:
        print("Anulowano.")
else:
    print("Folder nie istnieje, nic do usuniecia.")
